# CS462 – Thai Handwritten Two-Digit Classification (๒๑–๒๙)
**Train CNN จาก dataset ลายมือจริง ~388 รูป/digit**

**วิธีใช้:**
1. Runtime → Change runtime type → **GPU (T4)**
2. อัปโหลดโฟลเดอร์ `thai-handwriting-number-master` ขึ้น Google Drive
3. รัน Cell ตามลำดับ

---

## 0. Install & Mount Drive

จัดทำโดย
1) ธนพงศ์ ชูแก้ว 1650702960 
2) ชนภูมิ กลมเกลี้ยง 1650702986

In [ ]:
import subprocess, sys
for pkg in ['pillow', 'matplotlib', 'scikit-learn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies ready')

In [ ]:
# Mount Google Drive (รัน cell นี้แล้วกด Allow)
from google.colab import drive
drive.mount('/content/drive')

# ── แก้ path ให้ตรงกับที่วางไฟล์ใน Drive ──
DATASET_ROOT = '/content/drive/MyDrive/thai-handwriting-number-master/data/raw/thai-handwriting-number.appspot.com'

import os
if os.path.exists(DATASET_ROOT):
    folders = [f for f in os.listdir(DATASET_ROOT) if f.isdigit()]
    print(f'Found {len(folders)} digit folders: {sorted(folders, key=int)}')
else:
    print(f'[ERROR] Path not found: {DATASET_ROOT}')
    print('แก้ DATASET_ROOT ให้ตรงกับ path ใน Google Drive ของคุณ')

## 1. Config

In [ ]:
# Thai digit Unicode map (folder number → Thai character)
THAI_DIGITS = {1:'๑', 2:'๒', 3:'๓', 4:'๔', 5:'๕', 6:'๖', 7:'๗', 8:'๘', 9:'๙', 0:'๐'}

# ๒๑–๒๙: first digit always ๒ (folder 2), second digit ๑–๙ (folders 1–9)
CLASSES = [f'๒{THAI_DIGITS[d]}' for d in range(1, 10)]  # ['๒๑','๒๒',...,'๒๙']
SECOND_DIGIT_FOLDERS = list(range(1, 10))                # [1,2,...,9]
NUM_CLASSES = len(CLASSES)                               # 9

IMG_SIZE        = 64   # รูปขาออก model
SAMPLES_PER_CLASS = 500  # สุ่มจับคู่จาก 389×388 combinations
EPOCHS          = 200
BATCH_SIZE      = 32
MODEL_NAME      = 'thai_digit_model.h5'

# aliases (กันพลาดถ้า cell อื่นใช้ชื่อนี้)
CATEGORIES = CLASSES
DATASET_DIR = DATASET_ROOT

print(f'Classes ({NUM_CLASSES}): {CLASSES}')
print(f'IMG: {IMG_SIZE}x{IMG_SIZE} | Samples/class: {SAMPLES_PER_CLASS} | Total: {NUM_CLASSES*SAMPLES_PER_CLASS}')

## 2. Imports

In [ ]:
import os, glob, json, random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from PIL import Image, ImageFilter
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, ConfusionMatrixDisplay,
                             precision_recall_fscore_support, accuracy_score)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# โหลด Thai font สำหรับ matplotlib
!mkdir -p fonts
FONT_URL = 'https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf'
FONT_PATH = 'fonts/Sarabun.ttf'
if not os.path.exists(FONT_PATH):
    !wget -q -O {FONT_PATH} {FONT_URL}
fm.fontManager.addfont(FONT_PATH)
thai_fp = fm.FontProperties(fname=FONT_PATH)
plt.rcParams['font.family'] = thai_fp.get_name()
print(f'Thai font ready: {thai_fp.get_name()}')

## 3. โหลด Dataset และสร้างรูปสองหลัก

**แนวทาง:** สุ่มจับคู่รูป ๒ (folder 2) กับรูป ๑–๙ (folder 1–9)  
วางเรียงแนวนอน → resize เป็น 64×64  

```
┌──────────┬──────────┐
│  ๒       │  ๑       │  →  label = ๒๑
│ (32×64)  │ (32×64)  │
└──────────┴──────────┘
       64×64
```

In [ ]:
def load_folder(folder_num):
    """
    โหลดรูปจาก folder และแปลงเป็น numpy array
    Raw images: 300×300 RGBA, พื้นขาว/เทา ตัวเลขดำ
    → แปลงเป็น grayscale แล้ว invert ให้เป็นพื้นดำ ตัวเลขขาว
    (ตรงกับที่ backend ทำกับ canvas ก่อนส่ง model)
    """
    folder_path = os.path.join(DATASET_ROOT, str(folder_num))
    images = []
    for fpath in glob.glob(os.path.join(folder_path, '*.png')):
        img = Image.open(fpath).convert('RGBA')
        # composite บน background ขาวเพื่อแปลง alpha
        bg = Image.new('L', img.size, 255)
        bg.paste(img.convert('L'))
        arr = np.array(bg, dtype=np.float32)
        arr = 255.0 - arr  # invert: พื้นดำ ตัวเลขขาว
        images.append(arr)
    print(f'  Folder {folder_num} ({THAI_DIGITS[folder_num]}): {len(images)} images')
    return images


print('Loading single-digit images...')
digit2_imgs = load_folder(2)  # ๒ — ใช้เป็นหลักแรกทุก class
digit_imgs  = {}              # ๑–๙ — ใช้เป็นหลักที่สอง
for d in SECOND_DIGIT_FOLDERS:
    digit_imgs[d] = load_folder(d)
print(f'\nโหลดสำเร็จ | ๒: {len(digit2_imgs)} | ๑–๙: {[len(v) for v in digit_imgs.values()]}')

In [ ]:
def augment_digit(arr):
    """Augment รูปตัวเลขเดี่ยวก่อนนำมาต่อกัน"""
    img = Image.fromarray(arr.astype(np.uint8))
    img = img.rotate(random.uniform(-12, 12), fillcolor=0)
    if random.random() > 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))
    return np.array(img, dtype=np.float32)


def make_combined(img2_arr, imgX_arr):
    """
    วางรูป ๒ และรูป X เรียงแนวนอน → resize เป็น IMG_SIZE x IMG_SIZE
    """
    half = IMG_SIZE // 2  # 32 px

    img2 = augment_digit(img2_arr.copy())
    imgX = augment_digit(imgX_arr.copy())

    # resize แต่ละหลักเป็น 32×64
    pil2 = Image.fromarray(img2.astype(np.uint8)).resize((half, IMG_SIZE), Image.LANCZOS)
    pilX = Image.fromarray(imgX.astype(np.uint8)).resize((half, IMG_SIZE), Image.LANCZOS)

    # ต่อแนวนอน → 64×64
    combined = np.concatenate([np.array(pil2, dtype=np.float32),
                               np.array(pilX, dtype=np.float32)], axis=1)

    # noise augmentation
    if random.random() > 0.4:
        combined = np.clip(combined + np.random.normal(0, random.uniform(3, 12), combined.shape), 0, 255)

    return combined


print('Building two-digit dataset (๒๑–๒๙)...')
X, y = [], []
for label_idx, d in enumerate(SECOND_DIGIT_FOLDERS):  # d = 1..9
    imgs_d = digit_imgs[d]
    for _ in range(SAMPLES_PER_CLASS):
        img2 = random.choice(digit2_imgs)
        imgX = random.choice(imgs_d)
        combined = make_combined(img2, imgX)
        X.append(combined)
        y.append(label_idx)
    print(f'  {CLASSES[label_idx]}: {SAMPLES_PER_CLASS} samples')

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y = np.array(y)
print(f'\nDataset: {X.shape} | Labels: {np.unique(y)} | Classes: {CLASSES}')

In [ ]:
# แสดงตัวอย่างรูปสองหลักที่สร้าง
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 2, 5))
for c in range(NUM_CLASSES):
    idxs = np.where(y == c)[0]
    for r in range(2):
        ax = axes[r, c]
        ax.imshow(X[idxs[r]].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
        ax.set_title(CLASSES[c], fontproperties=thai_fp, fontsize=14)
        ax.axis('off')
plt.suptitle('ตัวอย่างรูปสองหลักที่สร้างจาก dataset จริง', fontproperties=thai_fp, fontsize=14)
plt.tight_layout()
plt.show()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

y_train_cat = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat  = keras.utils.to_categorical(y_test,  NUM_CLASSES)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train per class: {np.bincount(y_train)}')
print(f'Test  per class: {np.bincount(y_test)}')

## 5. Build & Train CNN

In [ ]:
def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=NUM_CLASSES):
    """
    CNN ที่ออกแบบให้รู้จักเลขสองหลัก
    - ชั้นแรก: เรียนรู้ feature จากทั้งสองหลัก
    - GlobalAveragePooling: ลด overfitting
    - label_smoothing: ป้องกัน overconfident
    """
    model = keras.Sequential([
        # Block 1 – low-level features
        layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 2 – mid-level features
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 3 – high-level features
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),

        # Classifier
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    return model

model = build_cnn()
model.summary()

In [ ]:
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=12,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.12,
    shear_range=0.08
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(
        'best_model.h5', save_best_only=True, monitor='val_accuracy', verbose=1
    )
]

history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_test, y_test_cat),
    callbacks=callbacks,
    verbose=1
)

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history.history['accuracy'],     label='Train', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val',   lw=2)
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(history.history['loss'],         label='Train', lw=2)
ax2.plot(history.history['val_loss'],     label='Val',   lw=2)
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 7. Evaluation

In [ ]:
# โหลด best model
if os.path.exists('best_model.h5'):
    model = keras.models.load_model('best_model.h5')
    print('Loaded best_model.h5')

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
print(f'\n{"="*55}')
print(f'  Overall Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'{"="*55}\n')
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))

In [ ]:
prec, rec, f1, sup = precision_recall_fscore_support(y_test, y_pred, average=None)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))
auc_scores = []
for i in range(NUM_CLASSES):
    fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    auc_scores.append(auc(fpr_i, tpr_i))
auc_m = np.mean(auc_scores)

fig = plt.figure(figsize=(18, 14))
fig.suptitle('สรุปผลประสิทธิภาพโมเดล ๒๑–๒๙',
             fontsize=18, fontproperties=thai_fp, y=0.98)

# Confusion Matrix
ax1 = fig.add_subplot(2, 2, 1)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
disp.plot(ax=ax1, cmap='Blues', values_format='d', colorbar=False)
for lb in ax1.get_xticklabels() + ax1.get_yticklabels():
    lb.set_fontproperties(thai_fp)
ax1.set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# Precision / Recall / F1
ax2 = fig.add_subplot(2, 2, 2)
x_pos, w = np.arange(NUM_CLASSES), 0.25
ax2.bar(x_pos - w, prec, w, label='Precision', color='#4e79a7')
ax2.bar(x_pos,     rec,  w, label='Recall',    color='#f28e2b')
ax2.bar(x_pos + w, f1,   w, label='F1-Score',  color='#59a14f')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(CLASSES, fontproperties=thai_fp, fontsize=10)
ax2.set_ylim(0, 1.2); ax2.set_ylabel('Score')
ax2.set_title('Precision / Recall / F1 per Class', fontsize=13, fontweight='bold')
ax2.legend()
for i in range(NUM_CLASSES):
    ax2.text(i - w, prec[i] + 0.02, f'{prec[i]:.2f}', ha='center', fontsize=7)
    ax2.text(i,     rec[i]  + 0.02, f'{rec[i]:.2f}',  ha='center', fontsize=7)
    ax2.text(i + w, f1[i]   + 0.02, f'{f1[i]:.2f}',   ha='center', fontsize=7)

# ROC
ax3 = fig.add_subplot(2, 2, 3)
colors = plt.cm.Set1(np.linspace(0, 1, NUM_CLASSES))
for i in range(NUM_CLASSES):
    fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    ax3.plot(fpr_i, tpr_i, color=colors[i], lw=2,
             label=f'{CLASSES[i]} (AUC={auc_scores[i]:.3f})')
ax3.plot([0, 1], [0, 1], 'k--', lw=1)
ax3.set_xlabel('FPR'); ax3.set_ylabel('TPR')
ax3.set_title('ROC Curve (One-vs-Rest)', fontsize=13, fontweight='bold')
ax3.legend(loc='lower right', prop=thai_fp, fontsize=8)

# Metrics table
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')
table_data = [['Class', 'Precision', 'Recall', 'F1', 'AUC', 'N']]
for i in range(NUM_CLASSES):
    table_data.append([CLASSES[i], f'{prec[i]:.4f}', f'{rec[i]:.4f}',
                       f'{f1[i]:.4f}', f'{auc_scores[i]:.4f}', f'{int(sup[i])}'])
table_data.append(['Macro', f'{prec_m:.4f}', f'{rec_m:.4f}',
                   f'{f1_m:.4f}', f'{auc_m:.4f}', f'{int(sum(sup))}'])
tbl = ax4.table(cellText=table_data, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.0, 1.6)
for j in range(6):
    tbl[0, j].set_facecolor('#4e79a7')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
    tbl[NUM_CLASSES+1, j].set_facecolor('#e8e8e8')
    tbl[NUM_CLASSES+1, j].set_text_props(fontweight='bold')
ax4.set_title('Classification Metrics Summary', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('performance_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('=' * 55)
print(f'  Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
print(f'  Precision : {prec_m:.4f}  Recall: {rec_m:.4f}  F1: {f1_m:.4f}  AUC: {auc_m:.4f}')
print('=' * 55)

In [ ]:
# ตัวอย่าง prediction แบบสุ่ม
fig, axes = plt.subplots(2, 9, figsize=(18, 5))
idxs = np.random.choice(len(X_test), 18, replace=False)
for i, idx in enumerate(idxs):
    ax = axes[i // 9, i % 9]
    ax.imshow(X_test[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    pred = CLASSES[y_pred[idx]]
    true = CLASSES[y_test[idx]]
    color = 'green' if pred == true else 'red'
    ax.set_title(f'P:{pred}\nT:{true}', color=color, fontsize=9, fontproperties=thai_fp)
    ax.axis('off')
plt.suptitle('Sample Predictions  (green=correct, red=wrong)', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Save & Download

In [ ]:
model.save(MODEL_NAME)
print(f'Saved: {MODEL_NAME}  ({os.path.getsize(MODEL_NAME)/1024:.0f} KB)')

meta = {'classes': CLASSES, 'img_size': IMG_SIZE}
with open('class_labels.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('Saved: class_labels.json')
print(f'  classes: {CLASSES}')

try:
    from google.colab import files
    files.download(MODEL_NAME)
    files.download('class_labels.json')
    files.download('performance_summary.png')
except Exception:
    print('(ไม่ได้รันบน Colab — ไฟล์อยู่ใน directory ปัจจุบัน)')

## 9. วิธีใช้กับ Flask Backend

```bash
# วางไฟล์ที่ download มา
#   thai_digit_model.h5
#   class_labels.json
# ไว้ใน folder เดียวกับ app.py แล้วรัน:

pip install flask flask-cors tensorflow pillow numpy
python app.py
# → เปิด http://localhost:5000
```

### ทดสอบ
1. เปิดหน้าเว็บ → วาดเลข **๒๑–๒๙** บน canvas
2. กด **Predict** → ระบบทำนายและแสดง confidence ของทุก class
3. Admin Page → Upload model ใหม่ได้โดยไม่ต้อง restart

### เกณฑ์ความสำเร็จที่คาดหวัง
| Metric | เป้าหมาย |
|--------|----------|
| Accuracy | ≥ 85% |
| Macro F1 | ≥ 0.85 |
| AUC | ≥ 0.95 |

> **เคล็ดลับ accuracy:** ถ้าได้ต่ำกว่า 80% ให้เพิ่ม `SAMPLES_PER_CLASS` เป็น 700–800
> และเพิ่ม `EPOCHS` เป็น 300 แล้ว train ใหม่